# Week 3 follow-up: unfreeze vision encoder + more epochs (run on Kaggle GPU)

**Motivation:** Week 3's Grad-CAM analysis found real (not just noisy) gradient signal spread
onto background regions in several heatmaps. Week 2's fine-tuning only ever trained the *text
decoder* -- the vision encoder was frozen throughout, so its attention patterns are 100% the
original pretrained BLIP, untouched by our fine-tuning. This notebook tests whether unfreezing
part of the vision encoder sharpens visual grounding (not just caption fluency).

**Controlled comparison against config A** (lr=5e-5, vision frozen, 1 epoch, same 1500-image
training subset): this run changes exactly two things --
1. **Unfreeze the last 2 vision encoder layers** (out of 12) instead of freezing all of them
2. **3 epochs instead of 1**

Same training images, same eval images, same learning rate, same decoding options -- so any
difference in BLEU/ROUGE or Grad-CAM sharpness can be attributed to those two changes.

## Setup
1. New Kaggle Notebook, paste this file in.
2. **Settings -> Accelerator -> GPU** (T4 x2 or P100).
3. **Add Data** -> `adityajn105/flickr8k` (same dataset as before).
4. Run all cells (~15-20 min: 3 epochs is 3x config A's training time, same eval cost).
5. Download `week3_finetune_vision_unfrozen_results.json`/`.csv` and `config_D_checkpoint/` from
   the Output tab into this repo's `results/` folder.

### 1. Setup (same as Week 2's Kaggle notebook)

In [ ]:
import os
import json
import time
import random
import pandas as pd
import torch
import evaluate
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration

KAGGLE_INPUT_DIR = "/kaggle/input/flickr8k"
OUTPUT_DIR = "/kaggle/working"

CAPTIONS_PATH = os.path.join(KAGGLE_INPUT_DIR, "captions.txt")
IMG_DIR = os.path.join(KAGGLE_INPUT_DIR, "Images")

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU detected -- check Settings > Accelerator on Kaggle."
print(f"Using device: {device}")

MODEL_NAME = "Salesforce/blip-image-captioning-base"

df = pd.read_csv(CAPTIONS_PATH)
df.columns = ["image", "caption"]
print(f"Total captions: {len(df)}, unique images: {df['image'].nunique()}")

### 2. Train / eval split -- IDENTICAL to config A's split (same seed=42, same sizes)

Reusing the exact same split is what makes this a controlled comparison rather than a new,
unrelated experiment.

In [ ]:
random.seed(42)
all_images = df["image"].drop_duplicates().tolist()
random.shuffle(all_images)

eval_images = all_images[:100]
train_images = all_images[100:100 + 1500]

train_df = df[df["image"].isin(train_images)].groupby("image").head(2).reset_index(drop=True)
eval_refs = [df[df["image"] == img]["caption"].tolist() for img in eval_images]

print(f"Training pairs: {len(train_df)} (from {len(train_images)} images)")
print(f"Eval images: {len(eval_images)}")

### 3. Dataset / collator (unchanged)

In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)

class FlickrFineTuneDataset(Dataset):
    def __init__(self, dataframe, img_dir):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        return image, row["caption"]

def collate_fn(batch):
    images, captions = zip(*batch)
    inputs = processor(
        images=list(images), text=list(captions),
        padding="max_length", truncation=True, max_length=32, return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

train_dataset = FlickrFineTuneDataset(train_df, IMG_DIR)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
print(f"{len(train_loader)} batches/epoch")

### 4. Train + eval functions -- `unfreeze_last_n_vision_layers` is the new knob

BLIP-base's vision encoder has 12 transformer blocks
(`model.vision_model.encoder.layers[0..11]`). `unfreeze_last_n_vision_layers=2` keeps layers
0-9 frozen and makes layers 10-11 trainable, alongside the text decoder (always trainable).

In [ ]:
def train_model(lr, num_epochs=1, unfreeze_last_n_vision_layers=0):
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

    num_vision_layers = len(model.vision_model.encoder.layers)
    freeze_up_to = num_vision_layers - unfreeze_last_n_vision_layers
    for i, layer in enumerate(model.vision_model.encoder.layers):
        for p in layer.parameters():
            p.requires_grad = i >= freeze_up_to
    # embeddings/pre-layernorm stay frozen regardless -- only the top N transformer blocks unfreeze
    for p in model.vision_model.embeddings.parameters():
        p.requires_grad = False

    trainable = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    print(f"Trainable params: {n_trainable:,} (vision layers unfrozen: {unfreeze_last_n_vision_layers}/{num_vision_layers})")
    optimizer = torch.optim.AdamW(trainable, lr=lr)

    model.train()
    losses = []
    for epoch in range(num_epochs):
        for batch in tqdm(train_loader, desc=f"train lr={lr} epoch={epoch}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            losses.append(loss.item())
    return model, losses


@torch.no_grad()
def evaluate_model(model, decoding="greedy"):
    model.eval()
    gen_kwargs = {"max_new_tokens": 30}
    gen_kwargs["num_beams"] = 4 if decoding == "beam" else 1

    predictions = []
    for img_name in tqdm(eval_images, desc=f"eval decoding={decoding}"):
        image = Image.open(os.path.join(IMG_DIR, img_name)).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        out = model.generate(**inputs, **gen_kwargs)
        predictions.append(processor.decode(out[0], skip_special_tokens=True))

    bleu = evaluate.load("sacrebleu").compute(predictions=predictions, references=eval_refs)
    rouge = evaluate.load("rouge").compute(predictions=predictions, references=eval_refs)
    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"] * 100,
        "rouge2": rouge["rouge2"] * 100,
        "rougeL": rouge["rougeL"] * 100,
    }, predictions

### 5. Run config D (unfrozen vision, 3 epochs) + config E (same weights, beam search)

In [ ]:
results = {}

# Config D: unfreeze last 2 vision layers, lr=5e-5 (same as config A), 3 epochs, greedy
t0 = time.time()
model_d, losses_d = train_model(lr=5e-5, num_epochs=3, unfreeze_last_n_vision_layers=2)
metrics_d, preds_d = evaluate_model(model_d, decoding="greedy")
results["D_unfrozen2_3ep_greedy"] = {
    "lr": 5e-5, "decoding": "greedy", "unfrozen_vision_layers": 2, "epochs": 3,
    "final_train_loss": losses_d[-1], "train_time_s": time.time() - t0, **metrics_d,
}
model_d.save_pretrained(os.path.join(OUTPUT_DIR, "config_D_checkpoint"))

# Config E: same weights as D, beam search decoding (mirrors how C reused A's weights)
metrics_e, preds_e = evaluate_model(model_d, decoding="beam")
results["E_unfrozen2_3ep_beam"] = {
    "lr": 5e-5, "decoding": "beam", "unfrozen_vision_layers": 2, "epochs": 3,
    "final_train_loss": losses_d[-1], "train_time_s": results["D_unfrozen2_3ep_greedy"]["train_time_s"],
    **metrics_e,
}

results_df = pd.DataFrame(results).T
results_df

### 6. Save results for download

In [ ]:
with open(os.path.join(OUTPUT_DIR, "week3_finetune_vision_unfrozen_results.json"), "w") as f:
    json.dump(results, f, indent=2)

results_df.to_csv(os.path.join(OUTPUT_DIR, "week3_finetune_vision_unfrozen_results.csv"))

print("Compare against config A (lr=5e-5, vision frozen, 1 epoch): BLEU 27.2, ROUGE-L 50.5")
print("Compare against config C (config A weights + beam search): BLEU 28.9, ROUGE-L 52.2")
print()
print("Download week3_finetune_vision_unfrozen_results.json/csv AND config_D_checkpoint/")
print("from the Output tab into this repo's results/ folder.")